# Tune the NMF representation

Control panel for the NMF dictionary sweep. It wraps the command-line pipeline scripts (nothing is reimplemented here):

- [`tune_representation.py`](tune_representation.py) runs the Optuna sweep
- [`analyze_sweep.py`](analyze_sweep.py) reads the study and scores configs
- [`train_gap_model.py`](train_gap_model.py) / [`build_dict_codes.py`](build_dict_codes.py) apply the best trial

The sweep runs as a subprocess writing a resumable SQLite study, so you can stop and re-run this notebook to add trials, and analyze partial results any time.

**What is being tuned:** an NMF dictionary is fit on rasterized 1H/13C spectra. The sweep searches how spectra are turned into a matrix (peak widths, smoothing, intensity transform, modality balance) and how NMF factorizes it (components, solver, regularization), optimizing **held-out `gap_ev` R²** (primary) with **logP R²** as a second objective — the two define a Pareto front.

## The tunable options

Every knob below is searched per trial (ranges are the defaults in `tune_representation.py`'s `objective`). This table is the "different options available" reference.

**Spectrum → matrix (rasterization):**

| Knob | Range | Meaning |
|------|-------|---------|
| `n_components` | 10–120 (step 5) | NMF dictionary size (# motifs) |
| `h_sigma_ppm` | 0.01–0.10 | Gaussian width for 1H peaks |
| `c_sigma_ppm` | 0.25–2.0 | Gaussian width for 13C peaks |
| `h_width_scale` / `c_width_scale` | 0.1–1.0 / 0.5–2.0 | scale measured peak widths |
| `h_multiplicity` | True/False | expand 1H peaks by multiplicity class |
| `use_peak_width` | True/False | use measured widths vs. fixed sigma |
| `intensity_transform` | none / sqrt / cbrt / log1p / arcsinh | compress peak intensities |
| `h_modality_weight` | 0.1–10 (log) | balance of 1H vs 13C block in the matrix |

**NMF factorization:**

| Knob | Range | Meaning |
|------|-------|---------|
| `solver` | cd / mu | coordinate descent vs multiplicative update |
| `beta_loss` | frobenius / kullback-leibler | reconstruction loss (mu solver) |
| `alpha_W`, `alpha_H` | 1e-5–1e-2 (log) | L1/L2 regularization strength |
| `l1_ratio` | 0–1 | sparsity vs ridge mix |

**Guardrails reported per trial (not optimized directly):** held-out reconstruction error and functional-group micro-F1.

## Config

In [ ]:
from pathlib import Path
from nmrlib.data import resolve_dataset

# Dataset with raw h_nmr_peaks / c_nmr_peaks + gap_ev/logP labels
dataset = "alberts_10k"            # registry name or path

# Sweep controls (surfaced from tune_representation.py's argparse)
n_trials = 50                      # total trials the study should reach
sample = 12000                     # cap on molecules used per sweep
probe = "elasticnet"               # linear probe scoring each config: elasticnet | rf
seed = 3245

# Where the study + results live. The .db is the resumable Optuna storage;
# re-running only adds trials up to n_trials.
out_json = Path("sweeps/nmf_repr.json")
storage = out_json.with_suffix(".db")
study_name = f"nmf_repr_{probe}"

raw_data = resolve_dataset(dataset)
print(f"raw_data : {raw_data}")
print(f"storage  : {storage}")
print(f"study    : {study_name}")

## Run the sweep

Runs `tune_representation.py` as a subprocess. It's resumable: set `n_trials` higher and re-run to add trials to the same study. For a quick smoke test drop `n_trials` to 2–3. Long sweeps are better run in a terminal (or on the cloud — see [`docker/README.md`](docker/README.md)) with the exact command printed below.

In [ ]:
import subprocess, sys, shlex

cmd = [
    sys.executable, "tune_representation.py",
    "--raw-data", str(raw_data),
    "--sample", str(sample),
    "--n-trials", str(n_trials),
    "--probe", probe,
    "--seed", str(seed),
    "--out", str(out_json),
    "--storage", str(storage),
    "--study-name", study_name,
]
print("Command:\n ", " ".join(shlex.quote(c) for c in cmd), "\n")

# Set run=True to launch from the notebook; keep False to just copy the command.
run = False
if run:
    proc = subprocess.run(cmd, cwd=str(Path.cwd()))
    print("exit code:", proc.returncode)

## Analyze the study

Reads the SQLite study with `analyze_sweep`'s loaders and shows the completed trials, the Pareto front (gap R² vs logP R²), the best config per objective, and hyperparameter importances.

In [ ]:
import optuna
import pandas as pd
from analyze_sweep import load_completed, GAP, LOGP

study, trials = load_completed(storage, study_name)
print(f"{len(trials)} completed trials")
trials.sort_values(GAP, ascending=False).head(10)

In [ ]:
# Pareto front: non-dominated trials trading off gap_ev R2 vs logP R2
import matplotlib.pyplot as plt

pareto = {t.number for t in study.best_trials}
is_p = trials['number'].isin(pareto)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(trials.loc[~is_p, GAP], trials.loc[~is_p, LOGP], c='lightgray', s=25, label='dominated')
ax.scatter(trials.loc[is_p, GAP], trials.loc[is_p, LOGP], c='crimson', s=45, label='Pareto')
for _, r in trials[is_p].iterrows():
    ax.annotate(int(r['number']), (r[GAP], r[LOGP]), fontsize=8)
ax.set_xlabel('held-out gap_ev R\u00b2'); ax.set_ylabel('held-out logP R\u00b2')
ax.set_title('Sweep Pareto front'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Best trial for each objective, with full hyperparameters
best_gap = trials.sort_values(GAP, ascending=False).iloc[0]
best_logp = trials.sort_values(LOGP, ascending=False).iloc[0]
print(f"Best gap_ev R\u00b2: trial {int(best_gap['number'])}  gap={best_gap[GAP]:.4f}  logP={best_gap[LOGP]:.4f}")
print(f"Best logP  R\u00b2: trial {int(best_logp['number'])}  gap={best_logp[LOGP]:.4f}  logP={best_logp[LOGP]:.4f}")

param_cols = [c for c in trials.columns if c in {
    'n_components','h_sigma_ppm','c_sigma_ppm','h_width_scale','c_width_scale',
    'h_multiplicity','use_peak_width','intensity_transform','h_modality_weight',
    'solver','beta_loss','alpha_W','alpha_H','l1_ratio'}]
best_gap[param_cols]

In [ ]:
# Which knobs actually move each objective (Optuna fANOVA importances)
for label, target in [('gap_ev R\u00b2', 0), ('logP R\u00b2', 1)]:
    try:
        imp = optuna.importance.get_param_importances(study, target=lambda t: t.values[target])
        s = pd.Series(imp).sort_values(ascending=True)
        s.plot.barh(figsize=(6, 4), title=f'Hyperparameter importance for {label}')
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'{label}: importance unavailable ({e})')

## Apply the best trial

Once you've picked a trial, hand it to the training/dictionary scripts. The study stores every trial's full hyperparameters, so these read the config straight from `--storage` + `--trial`.

- **Train a final gap model** on the chosen representation:
  ```
  python train_gap_model.py --raw-data <raw> --storage <db> \
      --study-name <name> --trial <n> --cv 5 --out-dir <dir>
  ```
- **Build codes for another dataset** with the tuned dictionary
  (needs the trial's params as JSON — `train_gap_model.py` writes one, or   export `best_gap[param_cols]`):
  ```
  python build_dict_codes.py --fit-data <A> --apply-data <B> \
      --params-json <params.json> --out <B_with_codes.pkl>
  ```

The command below is prefilled from `best_gap`; set `run=True` to launch it.

In [ ]:
import shlex

trial_to_use = int(best_gap['number'])
train_cmd = [
    sys.executable, "train_gap_model.py",
    "--raw-data", str(raw_data),
    "--storage", str(storage),
    "--study-name", study_name,
    "--trial", str(trial_to_use),
    "--cv", "5",
    "--out-dir", "gap_model",
]
print(" ".join(shlex.quote(c) for c in train_cmd))

run = False
if run:
    subprocess.run(train_cmd, cwd=str(Path.cwd()))